# **Feature visualization: попросить сеть нарисовать то, что она ищет**

Практика к модулю [«Что сеть выучила: признаки и концепции»](https://ai-interpretability.school).

В уроке мы задавали сети обратный вопрос: не «как канал реагирует на эту картинку», а
**«какая картинка заставит канал реагировать сильнее всего»**. Здесь мы этот вопрос зададим
по-настоящему — и увидим, почему наивный ответ на него бесполезен.

Тетрадь идет на процессоре: две оптимизации занимают около четверти минуты.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

torch.manual_seed(0)

SIZE = 160          # сторона картинки, которую рисуем
LAYER = 'layer4'    # слой, в котором живет интересующий канал
CHANNEL = 12        # номер канала — меняйте и смотрите, что нарисуется
STEPS = 384         # шагов оптимизации

## 1. Что оптимизируем

Все как в уроке: сеть обучена и заморожена, меняется только картинка на входе. Нам нужны две
вещи — способ узнать активацию выбранного канала и способ дотянуться до нее градиентом.

Активацию снимаем хуком: он перехватывает выход слоя на прямом проходе, и нам не приходится
переписывать саму сеть.

In [ ]:
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)          # веса заморожены: меняться будет только картинка

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

captured = {}
getattr(model, LAYER).register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))


def activation(img, channel=CHANNEL):
    """Средняя активация канала. На вход — картинка в [0,1], нормировка внутри."""
    model((img - MEAN) / STD)
    return captured['a'][0, channel].mean()


def show(img, title):
    plt.figure(figsize=(3, 3))
    plt.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
    plt.title(title)
    plt.axis('off')
    plt.show()

## 2. Наивная оптимизация

Делаем ровно то, что описано в уроке, и ничего сверх: стартуем с шума, шагаем по градиенту
активации, повторяем несколько сотен раз.

Функции `jitter` и `total_variation` понадобятся дальше — пока `optimize` вызывается с
выключенными и сдвигами, и штрафом.

In [ ]:
def optimize(use_jitter, tv_weight, steps=STEPS, lr=0.08):
    """Градиентный подъем по входу: ищем картинку, на которой канал откликнется сильнее."""
    img = (torch.rand(1, 3, SIZE, SIZE) * 0.1 + 0.45).requires_grad_(True)
    opt = torch.optim.Adam([img], lr=lr)
    for _ in range(steps):
        view = jitter(img) if use_jitter else img
        loss = -activation(view) + tv_weight * total_variation(img)
        opt.zero_grad()
        loss.backward()                # градиент считается по картинке, а не по весам
        opt.step()
        with torch.no_grad():
            img.clamp_(0, 1)           # картинка обязана остаться картинкой: значения в [0,1]
    return img.detach()


def jitter(img, max_shift=12):
    """Сдвиг и небольшое изменение масштаба перед прогоном — transformation robustness."""
    dx, dy = torch.randint(-max_shift, max_shift + 1, (2,))
    img = torch.roll(img, (int(dy), int(dx)), dims=(2, 3))
    n = max(int(SIZE * (1 + 0.15 * (torch.rand(1).item() - 0.5))), 8)
    return F.interpolate(img, size=(n, n), mode='bilinear', align_corners=False)


def total_variation(img):
    """Полная вариация: средний перепад между соседними пикселями. Чем больше, тем «шумнее»."""
    return ((img[:, :, 1:, :] - img[:, :, :-1, :]).abs().mean()
            + (img[:, :, :, 1:] - img[:, :, :, :-1]).abs().mean())

In [ ]:
naive = optimize(use_jitter=False, tv_weight=0.0)
with torch.no_grad():
    a_naive = activation(naive).item()

show(naive, f'наивная оптимизация, активация: {a_naive:.1f}')
print(f'активация канала: {a_naive:.2f}')
print(f'средний перепад между соседними пикселями: {total_variation(naive).item():.3f}')

## 3. Проверка: мы нашли состязательный пример

Картинка выше похожа на телевизионные помехи, а активация у нее большая. Насколько большая —
вопрос не риторический, и ответить на него можно точно: сравним с настоящими изображениями,
то есть с теми, ради которых сеть и обучалась.

In [ ]:
tf = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
FILES = ['data/cat.jpg', 'data/hog.jpg', 'data/pig.png', 'data/cat_and_dog.jpg',
         'assets/tim-foster-w-X64-Gjbclg-unsplash.jpg']


def load(path):
    raw = urllib.request.urlopen(f'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/' + path, timeout=30).read()
    return tf(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)


real = {path: load(path) for path in FILES}
with torch.no_grad():
    scores = {path: activation(img).item() for path, img in real.items()}

for path, value in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'{value:7.2f}  {path}')

best = max(scores.values())
print(f'\nво столько раз активация шума выше лучшей настоящей картинки: {a_naive / best:.0f}')

**Задание 1.** Отношение получилось в десятки раз. Проверьте, что дело не в
конкретном канале: возьмите три-четыре других номера в том же слое и посчитайте отношение для
каждого. Сохраняется ли порядок величины?

Что это значит: максимум активации лежит там, где картинок из реального мира не бывает.
Оптимизация отработала честно — просто честный максимум оказался бесполезен.

In [ ]:
# Ваш код здесь

## 4. Что удерживает оптимизацию в области правдоподобных картинок

В уроке названы три приема, и здесь работают два из них:

- **сдвиги и масштабирование** (`jitter`) — на каждом шаге картинка слегка дрожит перед
  прогоном. Состязательный шум хрупок и разрушается от сдвига на пару пикселей, а устойчивый
  паттерн переживает дрожание;
- **штраф за перепады** (`total_variation`) — прямо говорим оптимизации, что гладкое лучше.

Третий прием, замену параметризации на частотную, здесь не берем: он требует отдельного
разговора про спектр, а первые два дают эффект уже сами по себе.

In [ ]:
tamed = optimize(use_jitter=True, tv_weight=0.3)
with torch.no_grad():
    a_tamed = activation(tamed).item()

show(tamed, f'со сдвигами и штрафом, активация: {a_tamed:.1f}')
print(f'активация канала: {a_tamed:.2f}  (было {a_naive:.2f})')
print(f'средний перепад между соседними пикселями: {total_variation(tamed).item():.3f}')

**Задание 2.** Активация упала, а картинка стала читаемой. Это не совпадение, а
цена: мы сузили область поиска, и внутри нее максимум ниже.

Разберите, кто из двух приемов что делает. Запустите `optimize` еще дважды — со сдвигами, но
без штрафа, и со штрафом, но без сдвигов, — и сравните четыре картинки вместе с их активациями
и перепадами. Какой из приемов убирает шум, а какой почти не влияет?

In [ ]:
# Ваш код здесь

## 5. Рядом с нарисованным — настоящее

В уроке сказано: полезная практика — смотреть оптимизированную картинку вместе с датасетными
примерами. Оптимизация показывает, что канал ищет «в идеале»; настоящие снимки показывают, что
он находит на практике. **Расхождение между ними — само по себе диагноз.**

In [ ]:
top = sorted(scores.items(), key=lambda kv: -kv[1])[:3]

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(tamed[0].permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title(f'со сдвигами и штрафом, активация\n{a_tamed:.1f}')
for ax, (path, value) in zip(axes[1:], top):
    ax.imshow(real[path][0].permute(1, 2, 0).numpy())
    ax.set_title(f'{path.split("/")[-1]}\n{value:.2f}')
for ax in axes:
    ax.axis('off')
plt.show()

**Задание 3.** Наш набор настоящих картинок крошечный — семь штук, и ни одна не
подбиралась под этот канал. Возьмите два-три своих изображения по адресу и посмотрите, что
активирует канал сильнее.

Вопрос, ради которого все затевалось: похоже ли нарисованное на то, что канал реально находит?
Если непохоже — какой из двух ответов вы понесете заказчице и почему?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Оптимизация входа честно находит максимум — и он бесполезен.** Активация шума оказалась
  в десятки раз выше, чем у любой настоящей картинки. Это не поломка метода, а свойство
  пространства: осмысленные изображения занимают в нем ничтожную долю.
- **Регуляризация — не украшение, а условие осмысленности.** Без нее метод дает состязательный
  пример. Со сдвигами активация падает, и это правильная цена: мы ищем максимум внутри области
  правдоподобных картинок, а не по всему пространству.
- **Картинка зависит от процедуры.** Смените число шагов, шаг оптимизации, вес штрафа или
  стартовый шум — получите другой результат. Мы видим один из максимумов, отобранный нашими же
  ограничениями, а не «истинный образ канала».
- **Одна картинка на канал — сильное упрощение.** Если канал полисемантичен, оптимизация выдаст
  смесь. Об этом — урок «Одна единица — одна концепция?».